In [ ]:
# Function Calling 
# - Provides specific instructions for finding information
# - Prioritizes queries for precise results and desired format
# - Adds control and encapsulates queries for better structure and predictability


In [53]:
import os 
from openai import AzureOpenAI
import json

from pprint import pprint

client = AzureOpenAI(
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

In [62]:
%run interact_csv_and_sql_data/sql_helpers.py

In [56]:
# 1 - using an example -
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location. 
    The default unit when not specified is fahrenheit"""
    
    if "new york" in location.lower():
        return json.dumps(
            {"location": "New York", "temperature": "10", "unit": unit}
        )
    elif "san francisco" in location.lower():
        return json.dumps(
            {"location": "San Francisco", "temperature": "20", "unit": unit}
        )
    elif "las vegas" in location.lower():
        return json.dumps(
            {"location": "Las Vegas", "temperature": "30", "unit": unit}
        )
    else:
        return json.dumps(
            {"location": location, "temperature": "unknown"}
        )

get_current_weather("New York")

'{"location": "New York", "temperature": "10", "unit": "fahrenheit"}'

In [57]:
# 1.1 - define the tools 
messages = [
    {"role": "user",
     "content": """What's the weather like in San Francisco,
                   New York, and Las Vegas?"""
    }
]

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": """Get the current weather in a given
                              location.The default unit when not
                              specified is fahrenheit""",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": """The city and state,
                                        e.g. San Francisco, CA""",
                    },
                    "unit": {
                        "type": "string",
                        "default":"fahrenheit",
                        "enum": [ "fahrenheit", "celsius"],
                        "description": """The messuring unit for
                                          the temperature.
                                          If not explicitly specified
                                          the default unit is 
                                          fahrenheit"""
                    },
                },
                "required": ["location"],
            },
        },
    }
]

In [58]:
# 1.2 - use the function calling 
# is the method that asks the chat model to produce a response, optionally including tool-calling behavior.
response = client.chat.completions.create(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    messages=messages,
    tools=tools,
    tool_choice="auto", 
)

response_message = response.choices[0].message
tool_calls = response_message.tool_calls

print(response_message)
print("-------------")
print(tool_calls)

if tool_calls:
    
    available_functions = {
        "get_current_weather": get_current_weather,
    } 
    messages.append(response_message)  
    
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        function_to_call = available_functions[function_name]
        function_args = json.loads(tool_call.function.arguments)
        function_response = function_to_call(
            location=function_args.get("location"),
            unit=function_args.get("unit"),
        )
        messages.append(
            {
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": function_response,
            }
        )  
    print("-------------")
    pprint(messages)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_rLJzl9qS2Z8NilIJ6UhJ35D9', function=Function(arguments='{"location": "San Francisco, CA"}', name='get_current_weather'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_2oBwO2YkHEuryMlaT1cMgBBH', function=Function(arguments='{"location": "New York, NY"}', name='get_current_weather'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_y0LK5M4ZQIp1j4K4gIQYAjaU', function=Function(arguments='{"location": "Las Vegas, NV"}', name='get_current_weather'), type='function')])
-------------
[ChatCompletionMessageFunctionToolCall(id='call_rLJzl9qS2Z8NilIJ6UhJ35D9', function=Function(arguments='{"location": "San Francisco, CA"}', name='get_current_weather'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_2oBwO2YkHEuryMlaT1cMgBBH', function=Function(arguments='{"location": "New 

In [59]:
second_response = client.chat.completions.create(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    messages=messages,
)

pprint(second_response)

ChatCompletion(id='chatcmpl-Dm2QIS7dRzQgEKhhaaMgDriCR0lv2', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Current temperatures:\n\n- San Francisco: 20°\n- New York: 10°\n- Las Vegas: 30°\n\nThe temperature unit wasn’t provided by the weather service.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1780341430, model='gpt-5.4-2026-03-05', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=39, prompt_tokens=163, total_tokens=202, completion_tokens_deta

In [60]:
# 2 - Using our SQL Database 
from pathlib import Path
from sqlalchemy import create_engine
import pandas as pd

csv_path = Path("interact_csv_and_sql_data")/ "all-states-history.csv"
db_path = Path("interact_csv_and_sql_data")/ "db" / "test.db"

db_path.parent.mkdir(parents=True, exist_ok=True)
engine = create_engine(f'sqlite:///{db_path.resolve()}')

df = pd.read_csv(csv_path).fillna(value=0)
df.to_sql(
    "all_states_history",
    con=engine,
    if_exists="replace",
    index=False
)

20780

In [63]:
# test func
# In Alaska, there were 3 people hospitalised on 2021-03-05
get_hospitalized_increase_for_state_on_date(engine, "AK","2021-03-05")

{'date': '2021-03-05', 'hospitalizedIncrease': 3}

In [67]:
# create user message 
messages = [
    {"role": "user",
     "content": """ how many hospitalized people we had in Alaska
                    the 2021-03-05?"""
    }
]


In [68]:
response = client.chat.completions.create(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    messages=messages,
    tools=tools_sql,
    tool_choice="auto",
)

response_message = response.choices[0].message
tool_calls = response_message.tool_calls

if tool_calls:
    print (tool_calls)
    
    available_functions = {
        "get_positive_cases_for_state_on_date": get_positive_cases_for_state_on_date,
        "get_hospitalized_increase_for_state_on_date":get_hospitalized_increase_for_state_on_date
    }  
    messages.append(response_message)  
   
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        function_to_call = available_functions[function_name]
        function_args = json.loads(tool_call.function.arguments)
        function_response = function_to_call(
            engine=engine,
            state_abbr=function_args.get("state_abbr"),
            specific_date=function_args.get("specific_date"),
        )
        messages.append(
            {
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": str(function_response),
            }
        ) 
    print(messages)

[ChatCompletionMessageFunctionToolCall(id='call_SFBHXKLZEgd9o2fhMJOAQsOS', function=Function(arguments='{"state_abbr":"AK","specific_date":"2021-03-05"}', name='get_hospitalized_increase_for_state_on_date'), type='function')]
[{'role': 'user', 'content': ' how many hospitalized people we had in Alaska\n                    the 2021-03-05?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_SFBHXKLZEgd9o2fhMJOAQsOS', function=Function(arguments='{"state_abbr":"AK","specific_date":"2021-03-05"}', name='get_hospitalized_increase_for_state_on_date'), type='function')]), {'tool_call_id': 'call_SFBHXKLZEgd9o2fhMJOAQsOS', 'role': 'tool', 'name': 'get_hospitalized_increase_for_state_on_date', 'content': "{'date': '2021-03-05', 'hospitalizedIncrease': 3}"}]


In [66]:
second_response = client.chat.completions.create(
            model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
            messages=messages,
        )
print (second_response)

ChatCompletion(id='chatcmpl-Dm2Srg9pLWCp6KN05fQl1DpZ7UdUT', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='In Alaska on 2021-03-05, the hospitalized increase was 3.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1780341589, model='gpt-5.4-2026-03-05', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=22, prompt_tokens=100, total_tokens=122, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tok